In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import geopandas as gpd
import pandas as pd
import folium
import json
import struct
import branca.colormap as cm
import math

# =========================
# 1. COORDINATE TRANSFORMATION
# =========================

def stateplane_to_latlon(x_feet, y_feet):
    feet_to_meters = 0.3048006096012192
    x_m = x_feet * feet_to_meters
    y_m = y_feet * feet_to_meters
    false_easting = 1968500.0 * feet_to_meters
    false_northing = 0.0
    central_meridian = -84.5
    lat_origin = 29.0
    std_parallel_1 = 29.58333333333333
    std_parallel_2 = 30.75
    a = 6378137.0
    e = 0.08181919084262157
    phi0 = math.radians(lat_origin)
    phi1 = math.radians(std_parallel_1)
    phi2 = math.radians(std_parallel_2)
    lambda0 = math.radians(central_meridian)

    def m_calc(phi):
        return math.cos(phi) / math.sqrt(1 - e**2 * math.sin(phi)**2)

    def t_calc(phi):
        return math.tan(math.pi/4 - phi/2) / ((1 - e*math.sin(phi))/(1 + e*math.sin(phi)))**(e/2)

    m1 = m_calc(phi1)
    m2 = m_calc(phi2)
    t0 = t_calc(phi0)
    t1 = t_calc(phi1)
    t2 = t_calc(phi2)
    n = (math.log(m1) - math.log(m2)) / (math.log(t1) - math.log(t2))
    F = m1 / (n * t1**n)
    rho0 = a * F * t0**n
    x_adj = x_m - false_easting
    y_adj = y_m - false_northing
    rho = math.sqrt(x_adj**2 + (rho0 - y_adj)**2) * (-1 if n < 0 else 1)
    theta = math.atan2(x_adj, rho0 - y_adj)
    t = (rho / (a * F))**(1/n)
    phi = math.pi/2 - 2 * math.atan(t)
    for _ in range(10):
        phi_new = math.pi/2 - 2 * math.atan(t * ((1 - e*math.sin(phi))/(1 + e*math.sin(phi)))**(e/2))
        if abs(phi_new - phi) < 1e-10:
            break
        phi = phi_new
    lambda_val = theta / n + lambda0
    return math.degrees(lambda_val), math.degrees(phi)


def albers_to_latlon(x_m, y_m):
    false_easting = 400000.0
    false_northing = 0.0
    central_meridian = -84.0
    lat_origin = 24.0
    std_parallel_1 = 24.0
    std_parallel_2 = 31.5
    a = 6378137.0
    e = 0.08181919084262157
    e_sq = e * e
    phi0 = math.radians(lat_origin)
    phi1 = math.radians(std_parallel_1)
    phi2 = math.radians(std_parallel_2)
    lambda0 = math.radians(central_meridian)

    def q_calc(phi):
        sin_phi = math.sin(phi)
        return (1 - e_sq) * (
            sin_phi / (1 - e_sq * sin_phi**2) -
            (1 / (2 * e)) * math.log((1 - e * sin_phi) / (1 + e * sin_phi))
        )

    def m_calc(phi):
        return math.cos(phi) / math.sqrt(1 - e_sq * math.sin(phi)**2)

    m1 = m_calc(phi1)
    m2 = m_calc(phi2)
    q0 = q_calc(phi0)
    q1 = q_calc(phi1)
    q2 = q_calc(phi2)
    n = (m1**2 - m2**2) / (q2 - q1)
    C = m1**2 + n * q1
    rho0 = a * math.sqrt(C - n * q0) / n
    x_adj = x_m - false_easting
    y_adj = y_m - false_northing
    rho = math.sqrt(x_adj**2 + (rho0 - y_adj)**2)
    if n < 0:
        rho = -rho
    theta = math.atan2(x_adj, rho0 - y_adj)
    q = (C - (rho * n / a)**2) / n
    phi = math.asin(q / 2)
    for _ in range(10):
        sin_phi = math.sin(phi)
        term = (1 - e_sq * sin_phi**2)**2 / (2 * math.cos(phi))
        delta = term * (
            q / (1 - e_sq) -
            sin_phi / (1 - e_sq * sin_phi**2) +
            (1 / (2 * e)) * math.log((1 - e * sin_phi) / (1 + e * sin_phi))
        )
        phi += delta
        if abs(delta) < 1e-10:
            break
    lambda_val = lambda0 + theta / n
    return math.degrees(lambda_val), math.degrees(phi)


# =========================
# 2. SHAPEFILE READER FUNCTIONS
# =========================

def read_dbf(filename):
    with open(filename, 'rb') as f:
        header = f.read(32)
        num_records = struct.unpack('<I', header[4:8])[0]
        header_length = struct.unpack('<H', header[8:10])[0]
        record_length = struct.unpack('<H', header[10:12])[0]
        fields = []
        f.seek(32)
        while True:
            field_info = f.read(32)
            if field_info[0] == 0x0D:
                break
            field_name = field_info[:11].split(b'\x00')[0].decode('ascii')
            field_type = chr(field_info[11])
            field_length = field_info[16]
            fields.append((field_name, field_type, field_length))
        f.seek(header_length)
        records = []
        for _ in range(num_records):
            record = {}
            deleted = f.read(1)
            if deleted == b'*':
                f.read(record_length - 1)
                continue
            for field_name, field_type, field_length in fields:
                value = f.read(field_length).strip()
                if field_type == 'C':
                    record[field_name] = value.decode('ascii', errors='ignore').strip()
                elif field_type == 'N':
                    try:
                        record[field_name] = float(value) if b'.' in value else int(value)
                    except:
                        record[field_name] = 0
                elif field_type == 'F':
                    try:
                        record[field_name] = float(value) if value else 0.0
                    except:
                        record[field_name] = 0.0
                elif field_type == 'D':
                    record[field_name] = value.decode('ascii', errors='ignore').strip()
                else:
                    record[field_name] = value.decode('ascii', errors='ignore').strip()
            records.append(record)
    return records, fields


def read_shp(filename):
    with open(filename, 'rb') as f:
        f.read(100)
        geometries = []
        while True:
            record_header = f.read(8)
            if not record_header or len(record_header) < 8:
                break
            content_length = struct.unpack('>I', record_header[4:8])[0] * 2
            shape_type = struct.unpack('<I', f.read(4))[0]

            if shape_type == 0:
                geometries.append(None)
                continue
            elif shape_type == 1:
                x, y = struct.unpack('<2d', f.read(16))
                geometries.append({"type": "Point", "coordinates": [x, y]})
            elif shape_type in (3, 23):
                f.read(32)
                num_parts = struct.unpack('<I', f.read(4))[0]
                num_points = struct.unpack('<I', f.read(4))[0]
                parts = [struct.unpack('<I', f.read(4))[0] for _ in range(num_parts)]
                points = [list(struct.unpack('<2d', f.read(16))) for _ in range(num_points)]
                if shape_type == 23:
                    f.read(16)
                    f.read(8 * num_points)
                lines = []
                for i in range(num_parts):
                    start = parts[i]
                    end = parts[i + 1] if i + 1 < num_parts else num_points
                    lines.append(points[start:end])
                if len(lines) == 1:
                    geometries.append({"type": "LineString", "coordinates": lines[0], "needs_transform": True})
                else:
                    geometries.append({"type": "MultiLineString", "coordinates": lines, "needs_transform": True})
            elif shape_type == 5:
                f.read(32)
                num_parts = struct.unpack('<I', f.read(4))[0]
                num_points = struct.unpack('<I', f.read(4))[0]
                parts = [struct.unpack('<I', f.read(4))[0] for _ in range(num_parts)]
                points = [list(struct.unpack('<2d', f.read(16))) for _ in range(num_points)]
                rings = []
                for i in range(num_parts):
                    start = parts[i]
                    end = parts[i + 1] if i + 1 < num_parts else num_points
                    rings.append(points[start:end])
                needs_transform = any(abs(c) > 360 for ring in rings for pt in ring for c in pt)
                geometries.append({"type": "Polygon", "coordinates": rings, "needs_transform": needs_transform})
            else:
                f.read(content_length - 4)
                geometries.append(None)
    return geometries


def convert_tractce_to_csv_format(tractce):
    if not tractce:
        return None
    tractce_str = str(tractce).zfill(6)
    whole_int = int(tractce_str[:4])
    decimal = tractce_str[4:6]
    if decimal != '00':
        return f"{whole_int}.{decimal}"
    return str(whole_int)

def normalize_tract(tract_str):
    tract_str = str(tract_str).strip().replace('\xa0', ' ').strip()
    if tract_str.upper().startswith("CT"):
        tract_str = tract_str[2:]
    tract_str = tract_str.strip()
    try:
        if '.' in tract_str:
            whole, dec = tract_str.split('.', 1)
            whole = str(int(whole))
            dec_stripped = dec.rstrip('0') or '0'
            if dec_stripped == '0':
                tract_str = whole
            else:
                tract_str = f"{whole}.{dec_stripped}"
        else:
            tract_str = str(int(tract_str))
    except ValueError:
        pass
    return tract_str

# =========================
# 3. FILE PATHS
# =========================

BUS_STOPS_SHP    = "/content/drive/MyDrive/capstone/Random.shp"
BUS_STOPS_DBF    = "/content/drive/MyDrive/capstone/Random.dbf"
BUS_ROUTES_SHP   = "/content/drive/MyDrive/capstone/Spring2026_Weekday.shp"
BUS_ROUTES_DBF   = "/content/drive/MyDrive/capstone/Spring2026_Weekday.dbf"
CITY_LIMITS_SHP  = "/content/drive/MyDrive/capstone/par_citylm_2021.shp"
CITY_LIMITS_DBF  = "/content/drive/MyDrive/capstone/par_citylm_2021.dbf"
SHAPEFILE_SHP    = "/content/drive/MyDrive/capstone/tl_2020_12_tract.shp"
SHAPEFILE_DBF    = "/content/drive/MyDrive/capstone/tl_2020_12_tract.dbf"
CENSUS_DATA_FILE = "/content/drive/MyDrive/capstone/c_t_sheet_1.csv"
LIBRARY_FILE     = "/content/drive/MyDrive/capstone/aclib.csv"
VEHICLES_FILE    = "/content/drive/MyDrive/capstone/vehicles.csv"
COMMISSION_SHP   = "/content/drive/MyDrive/capstone/City_Commission_District_Export.shp"
COMMISSION_DBF   = "/content/drive/MyDrive/capstone/City_Commission_District_Export.dbf"


# =========================
# 4. LOAD DATA
# =========================

print("Loading data...")

# --- Bus stops ---
try:
    bus_geometries = read_shp(BUS_STOPS_SHP)
    bus_records, _ = read_dbf(BUS_STOPS_DBF)
    stops = []
    for i, record in enumerate(bus_records):
        if bus_geometries[i] and bus_geometries[i]['type'] == 'Point':
            coords = bus_geometries[i]['coordinates']
            stops.append({
                'BSID':       record.get('BSID', ''),
                'stop_name':  record.get('STOP_NAME', ''),
                'stop_desc':  record.get('DESCRIPTIO', ''),
                'Latitude':   coords[1],
                'Longitude':  coords[0],
                'STREET':     record.get('STREET', ''),
                'ACTIVE_INA': record.get('ACTIVE_INA', ''),
                'NO_BENCHES': record.get('NO_BENCHES', 0),
                'NO_SHELTER': record.get('NO_SHELTER', 0),
                'NO_BIKERAC': record.get('NO_BIKERAC', 0),
                'LANDING_PA': record.get('LANDING_PA', ''),
                'WAITING_PA': record.get('WAITING_PA', ''),
            })
    print(f"✓ Loaded {len(stops)} bus stops")
except FileNotFoundError as e:
    print(f"ERROR: {e}")
    stops = []

# --- Bus routes ---
try:
    route_geometries = read_shp(BUS_ROUTES_SHP)
    route_records, _ = read_dbf(BUS_ROUTES_DBF)
    routes = []
    for i, record in enumerate(route_records):
        if route_geometries[i]:
            geom = route_geometries[i]
            if geom.get('needs_transform', False):
                if geom['type'] == 'LineString':
                    geom['coordinates'] = [list(stateplane_to_latlon(x, y)) for x, y in geom['coordinates']]
                elif geom['type'] == 'MultiLineString':
                    geom['coordinates'] = [[list(stateplane_to_latlon(x, y)) for x, y in line]
                                           for line in geom['coordinates']]
                del geom['needs_transform']
            routes.append({
                'geometry':   geom,
                'route_id':   record.get('route_id', ''),
                'route_shor': record.get('route_shor', ''),
                'route_long': record.get('route_long', ''),
                'shape_id':   record.get('shape_id', ''),
                'start_time': record.get('start_time', ''),
                'end_time':   record.get('end_time', ''),
                'Colors':     record.get('Colors', ''),
                'State':      record.get('State', 0),
                'RouteType':  record.get('RouteType', 0),
                'length_mi':  record.get('length_mi', 0),
                'AM_freq':    record.get('AM_freq', 0),
                'Midday_fre': record.get('Midday_fre', 0),
                'PM_freq':    record.get('PM_freq', 0),
            })
    unique_routes = sorted(set(r['route_shor'] for r in routes if r['route_shor']))
    print(f"✓ Loaded {len(routes)} bus route segments ({len(unique_routes)} unique routes)")
    print(f"  Routes: {unique_routes}")
except FileNotFoundError as e:
    print(f"ERROR: {e}")
    routes = []

# --- Census FB data ---
try:
    census_data = pd.read_csv(CENSUS_DATA_FILE)
    census_data['FB']    = census_data['FB'].astype(str).str.replace(',', '').astype(int)
    census_data['Total'] = census_data['Total'].astype(str).str.replace(',', '').astype(int)
    census_data['FB_pct'] = (census_data['FB'] / census_data['Total'] * 100).round(2)
    census_data['TRACTCE'] = census_data['Census Tract'].str.replace('CT', '')
    print(f"✓ Loaded {len(census_data)} census tracts with FB data")
except FileNotFoundError:
    print(f"ERROR: Could not find '{CENSUS_DATA_FILE}'")
    census_data = None

# --- Gainesville city limits ---
try:
    city_geometries = read_shp(CITY_LIMITS_SHP)
    city_records, _ = read_dbf(CITY_LIMITS_DBF)
    gainesville_geom = None
    gainesville_acres = None
    for i, record in enumerate(city_records):
        if record.get('NAME', '').upper() == 'GAINESVILLE' and city_geometries[i]:
            geom = city_geometries[i]
            if geom.get('needs_transform', False):
                geom['coordinates'] = [
                    [list(albers_to_latlon(x, y)) for x, y in ring]
                    for ring in geom['coordinates']
                ]
                del geom['needs_transform']
            gainesville_geom = geom
            gainesville_acres = record.get('ACRES', 0)
            print(f"✓ Loaded Gainesville city limits ({gainesville_acres:.1f} acres = {gainesville_acres/640:.1f} sq mi)")
            break
    if not gainesville_geom:
        print("WARNING: Gainesville not found in city limits shapefile")
except FileNotFoundError as e:
    print(f"ERROR: {e}")
    gainesville_geom = None
    gainesville_acres = None

# --- Libraries ---
try:
    library_data = pd.read_csv(LIBRARY_FILE)
    libraries = library_data[['Branch', 'Latitude', 'Longitude']].to_dict('records')
    print(f"✓ Loaded {len(libraries)} library locations")
except FileNotFoundError:
    print(f"ERROR: Could not find '{LIBRARY_FILE}'")
    libraries = []
except Exception as e:
    print(f"ERROR loading library data: {e}")
    libraries = []

# --- Vehicle data ---
try:
    vehicles_raw = pd.read_csv(VEHICLES_FILE, index_col=0, encoding='latin-1')
    vehicles_raw.index = vehicles_raw.index.str.strip().str.replace('\xa0', ' ', regex=False).str.strip()
    vehicles_raw.columns = vehicles_raw.columns.astype(str).str.strip().str.replace('\xa0', ' ', regex=False).str.strip()
    vehicles_raw = vehicles_raw.apply(lambda col: pd.to_numeric(col.astype(str).str.replace(',', ''), errors='coerce')).fillna(0)

    total_row   = vehicles_raw.loc["Total"]
    has_veh_row = vehicles_raw.loc[["1", "2", "3"]].sum(axis=0)

    vehicle_data = {}
    for tract in total_row.index:
        tract_str = str(tract).strip()
        if tract_str.startswith("CT"):
            tract_str = tract_str[2:]
        total = total_row[tract]
        vehicle_data[tract_str] = round((has_veh_row[tract] / total) * 100, 2) if total > 0 else None

    print(f"✓ Loaded vehicle data for {len(vehicle_data)} census tracts")
    for tract, pct in list(vehicle_data.items())[:3]:
        if pct is not None:
            print(f"  Tract {tract}: {pct}% with at least one vehicle")
    print(f"  Sample vehicle_data keys: {list(vehicle_data.keys())[:5]}")

except FileNotFoundError:
    print(f"ERROR: Could not find '{VEHICLES_FILE}'")
    vehicle_data = {}
except KeyError as e:
    print(f"ERROR: Row label not found in vehicles.csv: {e}")
    print(f"  Available rows: {list(vehicles_raw.index)}")
    vehicle_data = {}

# --- Alachua County census tract shapefile ---
print("\nReading Alachua County shapefile...")
try:
    geometries = read_shp(SHAPEFILE_SHP)
    records, _ = read_dbf(SHAPEFILE_DBF)
    print(f"✓ Found {len(records)} census tracts in Florida shapefile")
except FileNotFoundError as e:
    print(f"ERROR: {e}")
    geometries = []
    records = []

alachua_features = []
fb_values = []

for i, record in enumerate(records):
    if record.get('STATEFP') == '12' and record.get('COUNTYFP') == '001':
        if geometries[i]:
            aland_sqmi  = record.get('ALAND', 0) / 2589988.11
            awater_sqmi = record.get('AWATER', 0) / 2589988.11
            tractce_raw = record.get('TRACTCE', '')
            tractce_conv = convert_tractce_to_csv_format(tractce_raw)

            fb_count = total_pop = fb_pct = None
            if census_data is not None and tractce_conv:
                match = census_data[census_data['TRACTCE'] == tractce_conv]
                if not match.empty:
                    fb_count  = int(match.iloc[0]['FB'])
                    fb_pct    = float(match.iloc[0]['FB_pct'])
                    total_pop = int(match.iloc[0]['Total'])
                    fb_values.append(fb_count)

            alachua_features.append({
                "type": "Feature",
                "geometry": geometries[i],
                "properties": {
                    "GEOID":       record.get('GEOID', ''),
                    "NAME":        record.get('NAME', ''),
                    "TRACTCE":     tractce_conv,
                    "TRACTCE_RAW": tractce_raw,
                    "ALAND_SQMI":  round(aland_sqmi, 2),
                    "AWATER_SQMI": round(awater_sqmi, 3),
                    "FB":          fb_count,
                    "FB_pct":      fb_pct,
                    "Total":       total_pop,
                    "veh_pct":     vehicle_data.get(tractce_conv),
                }
            })

print(f"✓ Filtered to {len(alachua_features)} Alachua County census tracts")
matched = sum(1 for f in alachua_features if f['properties']['FB'] is not None)
print(f"✓ Matched {matched} tracts with FB data")
veh_matched = sum(1 for f in alachua_features if f['properties']['veh_pct'] is not None)
print(f"✓ Matched {veh_matched} tracts with vehicle data")

unmatched = [f['properties']['TRACTCE'] for f in alachua_features if f['properties']['veh_pct'] is None]
if unmatched:
    print(f"  ⚠ {len(unmatched)} tracts unmatched: {sorted(unmatched)}")
    print(f"  Vehicle CSV keys (sample): {sorted(list(vehicle_data.keys()))[:10]}")

geojson_alachua = {"type": "FeatureCollection", "features": alachua_features}


# =========================
# 5. CREATE BASE MAP
# =========================

if stops:
    center_lat = sum(s['Latitude'] for s in stops) / len(stops)
    center_lon = sum(s['Longitude'] for s in stops) / len(stops)
else:
    center_lat, center_lon = 29.65163, -82.32483

m = folium.Map(location=[center_lat, center_lon], zoom_start=12, tiles="CartoDB positron")

# =========================
# 6. COLOR SCALES
# =========================

# FB (blue)
if fb_values:
    min_fb, max_fb = min(fb_values), max(fb_values)
    colormap = cm.LinearColormap(
        colors=['#f0f9ff','#bae6fd','#7dd3fc','#38bdf8','#0ea5e9','#0284c7','#0369a1','#075985','#0c4a6e'],
        vmin=min_fb, vmax=max_fb,
        caption='Foreign Born Population (FB)'
    )
    def get_color(fb_value):
        return '#cccccc' if fb_value is None else colormap(fb_value)
else:
    colormap = None
    def get_color(fb_value):
        return '#3388ff'

# Vehicle (green-purple)
no_veh_values = [v for v in vehicle_data.values() if v is not None]
if no_veh_values:
    min_noveh, max_noveh = min(no_veh_values), max(no_veh_values)
    vehicle_colormap = cm.LinearColormap(
        colors=['#f0f4f0', '#daeada', '#bcd8bc', '#97c297', '#72aa72', '#4e914e', '#347834', '#266026', '#1a6b1a'],
        vmin=min_noveh, vmax=max_noveh,
        caption='% Households with At Least One Vehicle'
    )
    def get_vehicle_color(tract_ce):
        pct = vehicle_data.get(tract_ce)
        return '#cccccc' if pct is None else vehicle_colormap(pct)
else:
    vehicle_colormap = None
    def get_vehicle_color(tract_ce):
        return '#cccccc'


# =========================
# 7. CENSUS TRACT LAYERS
# =========================

# --- Foreign Born Population ---
if alachua_features:
    fg_fb = folium.FeatureGroup(name="Foreign Born Population (FB)", show=True)
    folium.GeoJson(
        geojson_alachua,
        style_function=lambda f: {
            "fillColor": get_color(f['properties']['FB']),
            "color": "#000000", "weight": 1, "fillOpacity": 0.7,
        },
        popup=folium.GeoJsonPopup(
            fields=["TRACTCE", "FB", "FB_pct", "Total"],
            aliases=["Tract:", "Foreign Born:", "FB %:", "Total Pop:"],
            sticky=False, max_width=300,
        ),
    ).add_to(fg_fb)
    fg_fb.add_to(m)
    if colormap:
        colormap.add_to(m)

# --- Vehicle Access ---
if vehicle_data:
    fg_veh = folium.FeatureGroup(name="% of Households w/ at Least One Vehicle", show=False)
    folium.GeoJson(
        geojson_alachua,
        style_function=lambda f: {
            "fillColor": get_vehicle_color(f['properties']['TRACTCE']),
            "color": "#000000", "weight": 1, "fillOpacity": 0.7,
        },
        popup=folium.GeoJsonPopup(
            fields=["TRACTCE", "veh_pct", "FB", "Total"],
            aliases=["Tract:", "Has Vehicle (%):", "Foreign Born:", "Total Pop:"],
            sticky=False, max_width=300,
        ),
    ).add_to(fg_veh)
    fg_veh.add_to(m)
    if vehicle_colormap:
        vehicle_colormap.add_to(m)
    print("✓ Added Vehicle layer")


# =========================
# 8. BUS ROUTES
# =========================

if routes:
    bus_routes_group = folium.FeatureGroup(name="Bus Routes", show=False)
    routes_by_number = {}
    for route in routes:
        rn = route['route_shor']
        routes_by_number.setdefault(rn, []).append(route)

    for route in routes:
        geom = route['geometry']
        route_num = route['route_shor']
        color_hex = route.get('Colors', '').strip()
        route_color = f"#{color_hex}" if color_hex else '#3388ff'
        popup_html = f"""
        <div style="font-family: Arial; min-width: 250px;">
            <b style="font-size: 16px;">Route {route['route_shor']}</b><br>
            <b>{route['route_long']}</b><br>
            <hr style="margin: 5px 0;">
            <b>Shape ID:</b> {route['shape_id']}<br>
            <b>Hours:</b> {route['start_time']} - {route['end_time']}<br>
            <b>Length:</b> {route['length_mi']:.2f} miles<br>
            <hr style="margin: 5px 0;">
            <b>Frequency:</b><br>
            &bull; AM: {route['AM_freq']:.0f} min<br>
            &bull; Midday: {route['Midday_fre']:.0f} min<br>
            &bull; PM: {route['PM_freq']:.0f} min<br>
        </div>
        """
        folium.GeoJson(
            {"type": "Feature", "geometry": geom, "properties": {"color": route_color}},
            style_function=lambda x: {
                "color": x['properties']['color'], "weight": 4, "opacity": 0.8,
            },
            popup=folium.Popup(popup_html, max_width=300),
            tooltip=f"Route {route_num}: {route['route_long']}"
        ).add_to(bus_routes_group)
    bus_routes_group.add_to(m)


# =========================
# 9. BUS STOPS
# =========================

def point_to_line_distance(point_lon, point_lat, line_coords):
    min_dist = float('inf')
    for i in range(len(line_coords) - 1):
        x1, y1 = line_coords[i]
        x2, y2 = line_coords[i + 1]
        dx = (point_lon - x1) * 111320 * math.cos(math.radians(point_lat))
        dy = (point_lat - y1) * 110540
        seg_sq = ((x2-x1)*111320*math.cos(math.radians(y1)))**2 + ((y2-y1)*110540)**2
        if seg_sq == 0:
            dist = math.sqrt(dx**2 + dy**2)
        else:
            t = max(0, min(1, (dx*(x2-x1)*111320*math.cos(math.radians(y1)) + dy*(y2-y1)*110540) / seg_sq))
            dx2 = (point_lon - (x1+t*(x2-x1))) * 111320 * math.cos(math.radians(point_lat))
            dy2 = (point_lat - (y1+t*(y2-y1))) * 110540
            dist = math.sqrt(dx2**2 + dy2**2)
        min_dist = min(min_dist, dist)
    return min_dist

if stops and routes:
    stop_routes = {}
    proximity_threshold = 50

    for stop_idx, stop in enumerate(stops):
        serving = set()
        for route in routes:
            geom = route['geometry']
            rn = route['route_shor']
            if geom['type'] == 'LineString':
                if point_to_line_distance(stop['Longitude'], stop['Latitude'], geom['coordinates']) < proximity_threshold:
                    serving.add(rn)
            elif geom['type'] == 'MultiLineString':
                for line in geom['coordinates']:
                    if point_to_line_distance(stop['Longitude'], stop['Latitude'], line) < proximity_threshold:
                        serving.add(rn)
                        break
        stop_routes[stop_idx] = sorted(serving)

    multi_route_color = '#9333EA'
    bus_feature_group = folium.FeatureGroup(name="Bus Stops", show=False)

    for stop_idx, stop in enumerate(stops):
        serving_routes = stop_routes.get(stop_idx, [])
        amenities = []
        if stop.get('NO_BENCHES', 0) > 0: amenities.append(f"🪑 {stop['NO_BENCHES']} bench(es)")
        if stop.get('NO_SHELTER', 0) > 0: amenities.append(f"🏠 {stop['NO_SHELTER']} shelter(s)")
        if stop.get('NO_BIKERAC', 0) > 0: amenities.append(f"🚲 {stop['NO_BIKERAC']} bike rack(s)")
        amenities_html = "<br>".join(amenities) if amenities else "No amenities"

        ada_status = stop.get('LANDING_PA', 'Unknown')
        ada_icon = '🟢' if 'Compliant' in ada_status else ('🔴' if 'Non-Compliant' in ada_status else '⚪')
        routes_html = f"<b>Routes:</b> {', '.join(serving_routes)}<br>" if serving_routes else ""

        popup_html = f"""
        <div style="font-family: Arial; min-width: 200px;">
            <b style="font-size: 14px;">{stop['stop_name']}</b><br>
            <i>{stop['stop_desc']}</i><br>
            <hr style="margin: 5px 0;">
            {routes_html}
            <b>Street:</b> {stop.get('STREET', 'N/A')}<br>
            <b>Status:</b> {stop.get('ACTIVE_INA', 'Unknown')}<br>
            <b>ADA:</b> {ada_icon} {ada_status}<br>
            <hr style="margin: 5px 0;">
            <b>Amenities:</b><br>{amenities_html}
        </div>
        """

        if len(serving_routes) == 0:
            marker_color = fill_color = '#9CA3AF'
        elif len(serving_routes) == 1:
            color_hex = next(
                (r.get('Colors', '').strip() for r in routes if r['route_shor'] == serving_routes[0]), ''
            )
            marker_color = fill_color = f"#{color_hex}" if color_hex else '#3388ff'
        else:
            marker_color = fill_color = multi_route_color

        is_active = stop.get('ACTIVE_INA') == 'Active'
        opacity = 0.2 if not is_active else 0.5

        folium.CircleMarker(
            location=[stop["Latitude"], stop["Longitude"]],
            radius=3,
            popup=folium.Popup(popup_html, max_width=300),
            color=marker_color,
            fill=True,
            fillColor=fill_color,
            fillOpacity=opacity,
            opacity=opacity,
            weight=2,
        ).add_to(bus_feature_group)

    bus_feature_group.add_to(m)

elif stops:
    bus_feature_group = folium.FeatureGroup(name="Bus Stops", show=False)
    for stop in stops:
        amenities = []
        if stop.get('NO_BENCHES', 0) > 0: amenities.append(f"🪑 {stop['NO_BENCHES']} bench(es)")
        if stop.get('NO_SHELTER', 0) > 0: amenities.append(f"🏠 {stop['NO_SHELTER']} shelter(s)")
        if stop.get('NO_BIKERAC', 0) > 0: amenities.append(f"🚲 {stop['NO_BIKERAC']} bike rack(s)")
        amenities_html = "<br>".join(amenities) if amenities else "No amenities"

        ada_status = stop.get('LANDING_PA', 'Unknown')
        ada_icon = '🟢' if 'Compliant' in ada_status else ('🔴' if 'Non-Compliant' in ada_status else '⚪')

        popup_html = f"""
        <div style="font-family: Arial; min-width: 200px;">
            <b style="font-size: 14px;">{stop['stop_name']}</b><br>
            <i>{stop['stop_desc']}</i><br>
            <hr style="margin: 5px 0;">
            <b>Street:</b> {stop.get('STREET', 'N/A')}<br>
            <b>Status:</b> {stop.get('ACTIVE_INA', 'Unknown')}<br>
            <b>ADA:</b> {ada_icon} {ada_status}<br>
            <hr style="margin: 5px 0;">
            <b>Amenities:</b><br>{amenities_html}
        </div>
        """

        is_active = stop.get('ACTIVE_INA') == 'Active'

        folium.CircleMarker(
            location=[stop["Latitude"], stop["Longitude"]],
            radius=3,
            popup=folium.Popup(popup_html, max_width=300),
            color='green' if is_active else 'red',
            fill=True,
            fillColor='lightgreen' if is_active else 'lightcoral',
            fill_opacity=0.4,
        ).add_to(bus_feature_group)

    bus_feature_group.add_to(m)

# =========================
# 10. LIBRARIES
# =========================

if libraries:
    library_feature_group = folium.FeatureGroup(name="Libraries", show=False)
    for library in libraries:
        popup_html = f"""
        <div style="font-family: Arial; min-width: 150px;">
            <b style="font-size: 14px;">📚 {library['Branch']}</b><br>
            <i>Alachua County Library</i>
        </div>
        """
        folium.Marker(
            location=[library["Latitude"], library["Longitude"]],
            popup=folium.Popup(popup_html, max_width=250),
            tooltip=library['Branch'],
            icon=folium.Icon(color='red', icon='book', prefix='fa')
        ).add_to(library_feature_group)
    library_feature_group.add_to(m)


# =========================
# 11. INTERNET / COMPUTER LAYER
# =========================

INTERNET_FILE = "/content/drive/MyDrive/capstone/prop_computer_internet_household.csv"
try:
    internet_raw = pd.read_csv(INTERNET_FILE)
    internet_raw.columns = internet_raw.columns.str.strip()
    internet_data = {}
    computer_data = {}
    for _, row in internet_raw.iterrows():
        tract_str = normalize_tract(row['Tract'])
        internet_data[tract_str] = float(row['prop_broadband']) * 100 if pd.notna(row['prop_broadband']) else None
        computer_data[tract_str] = float(row['prop_computer']) * 100 if pd.notna(row['prop_computer']) else None
    print(f"✓ Loaded internet/computer data for {len(internet_data)} tracts")
except FileNotFoundError:
    print(f"ERROR: Could not find '{INTERNET_FILE}'")
    internet_data = {}
    computer_data = {}

for feature in alachua_features:
    tract_ce = feature['properties']['TRACTCE']
    feature['properties']['broadband_pct'] = internet_data.get(normalize_tract(tract_ce))
    feature['properties']['computer_pct']  = computer_data.get(normalize_tract(tract_ce))

inet_matched = sum(1 for f in alachua_features if f['properties']['broadband_pct'] is not None)
print(f"✓ Matched {inet_matched}/{len(alachua_features)} tracts with internet data")
unmatched_inet = [f['properties']['TRACTCE'] for f in alachua_features if f['properties']['broadband_pct'] is None]
print(f"  Unmatched TRACTCE values: {sorted(unmatched_inet)}")
print(f"  Internet CSV keys (all): {sorted(list(internet_data.keys()))}")

broadband_values = [v for v in internet_data.values() if v is not None]
if broadband_values:
    broadband_colormap = cm.LinearColormap(
        colors=['#ffffb7','#fff192','#ffea61','#ffdd3c','#ffd400','#c29200'],
        vmin=min(broadband_values), vmax=max(broadband_values),
        caption='% Households with Broadband (color) | Hatching = Computer %'
    )
    def get_broadband_color(tract_ce):
        pct = internet_data.get(normalize_tract(tract_ce))
        return '#cccccc' if pct is None else broadband_colormap(pct)
else:
    broadband_colormap = None
    def get_broadband_color(tract_ce):
        return '#cccccc'

if computer_data or internet_data:
    fg_inet = folium.FeatureGroup(name="Computer Access & Broadband", show=False)

    hatch_defs = {}
    pattern_ids = set()
    for feature in alachua_features:
        computer_pct = feature['properties'].get('computer_pct')
        if computer_pct is not None:
            spacing = max(4, min(50, int(50 - ((computer_pct - 75) / 25) * 46)))
            pid = f"hatch-{spacing}"
            if pid not in pattern_ids:
                hatch_defs[pid] = spacing
                pattern_ids.add(pid)

    patterns_svg = '<svg width="0" height="0" style="position:absolute"><defs>'
    for pid, spacing in hatch_defs.items():
        patterns_svg += f'''
        <pattern id="{pid}" patternUnits="userSpaceOnUse" width="{spacing}" height="{spacing}" patternTransform="rotate(45)">
          <line x1="0" y1="0" x2="0" y2="{spacing}" stroke="#000000" stroke-width="0.8"/>
        </pattern>'''
    patterns_svg += '</defs></svg>'
    m.get_root().html.add_child(folium.Element(patterns_svg))

    for feature in alachua_features:
        tract_ce = feature['properties']['TRACTCE']
        computer_pct = feature['properties'].get('computer_pct')
        broadband_pct = feature['properties'].get('broadband_pct')
        fill_color = get_broadband_color(tract_ce)

        # Base color fill with popup and highlight
        folium.GeoJson(
            feature,
            style_function=lambda f, fc=fill_color: {
                "fillColor": fc,
                "fillOpacity": 0.75,
                "color": "#333333",
                "weight": 1.5,
            },
            highlight_function=lambda f: {
                "fillColor": "#ffff00",
                "fillOpacity": 0.3,
                "color": "#333333",
                "weight": 2,
            },
            popup=folium.GeoJsonPopup(
                fields=["TRACTCE", "broadband_pct", "computer_pct", "FB", "Total"],
                aliases=["Census Tract:", "Broadband (%):", "Computer (%):", "Foreign Born:", "Total Pop:"],
                localize=True,
                sticky=False,
                style=(
                    "background-color: white;"
                    "border: 1px solid #333;"
                    "border-radius: 4px;"
                    "padding: 8px;"
                    "font-family: Arial;"
                    "font-size: 13px;"
                ),
                max_width=300,
            ),
        ).add_to(fg_inet)

        # Hatch overlay — pointer-events disabled so clicks pass through to base layer
        if computer_pct is not None:
            spacing = max(4, min(50, int(50 - ((computer_pct - 75) / 25) * 46)))
            pid = f"hatch-{spacing}"
            folium.GeoJson(
                feature,
                style_function=lambda f, pid=pid: {
                    "fillColor": f"url(#{pid})",
                    "fillOpacity": 0.5,
                    "color": "none",
                    "weight": 0,
                },
                interactive=False,   # <-- this disables mouse events on the hatch layer
            ).add_to(fg_inet)

    fg_inet.add_to(m)
    if broadband_colormap:
        broadband_colormap.add_to(m)
    print("✓ Added Computer Access & Broadband layer")

# =========================
# 12. RENT BURDEN & HOUSING DATA LAYERS
# =========================

HOUSING_FILE = "/content/drive/MyDrive/capstone/newhousingdata.csv"

GRAPI_ROWS = ["value_1", "value_2", "vaue_3", "value_4", "value_5", "value_6"]  # exclude value_7 (Not Computed)

row_aliases = {
    "value_1": "< 15% Total Inc",
    "value_2": "15-19% Total Inc",
    "vaue_3":  "20-24% Total Inc",
    "value_4": "25-29% Total Inc",
    "value_5": "30-34% Total Inc",
    "value_6": "35%+ Total Inc (Burdened)",
    "value_7": "Not Computed",
}

def load_wide_csv(filepath, label):
    """Load a CSV where rows = variables, columns = tract numbers."""
    raw = pd.read_csv(filepath, index_col=0)
    raw.index   = raw.index.astype(str).str.strip()
    raw.columns = raw.columns.astype(str).str.strip()
    for col in raw.columns:
        raw[col] = pd.to_numeric(raw[col].astype(str).str.replace(',', ''), errors='coerce')
    print(f"✓ Loaded {label} ({len(raw.columns)} tracts, {len(raw.index)} rows)")
    return raw

def build_tract_dict(raw):
    """Convert wide CSV to {tract_key: {row: float, ...}} dict."""
    result = {}
    for col in raw.columns:
        tract_key = normalize_tract(str(col).strip())
        result[tract_key] = {
            row: (float(raw.loc[row, col]) if pd.notna(raw.loc[row, col]) else None)
            for row in raw.index
        }
    return result

def attach_to_features(features, data_dict, row_labels, prefix):
    """Attach all row values from data_dict to feature properties."""
    for feature in features:
        tract_ce = normalize_tract(feature['properties']['TRACTCE'])
        d = data_dict.get(tract_ce, {})
        for row in row_labels:
            feature['properties'][f'{prefix}_{row}'] = d.get(row)
    matched = sum(1 for f in features if any(
        f['properties'].get(f'{prefix}_{row}') is not None for row in row_labels
    ))
    print(f"✓ Matched {matched}/{len(features)} tracts with {prefix} data")

def make_colormap(features, prop, colors, caption):
    """Build a LinearColormap from a feature property."""
    vals = [f['properties'].get(prop) for f in features if f['properties'].get(prop) is not None]
    if not vals:
        print(f"  WARNING: No values found for colormap property '{prop}'")
        return None
    return cm.LinearColormap(colors=colors, vmin=min(vals), vmax=max(vals), caption=caption)

def add_geojson_layer(m, geojson, color_fn, tooltip_fields, tooltip_aliases, name, colormap=None, popup_fields=None, popup_aliases=None):
    """Add a styled GeoJson FeatureGroup to the map."""
    fg = folium.FeatureGroup(name=name, show=False)

    popup = None
    if popup_fields and popup_aliases:
        popup = folium.GeoJsonPopup(
            fields=popup_fields,
            aliases=popup_aliases,
            localize=True,
            sticky=False,
            style=(
                "background-color: white;"
                "border: 1px solid #333;"
                "border-radius: 4px;"
                "padding: 8px;"
                "font-family: Arial;"
                "font-size: 13px;"
            ),
            max_width=320,
        )

    folium.GeoJson(
        geojson,
        style_function=lambda f, cfn=color_fn: {
            "fillColor": cfn(f['properties']['TRACTCE']),
            "color": "#000000", "weight": 1, "fillOpacity": 0.7,
        },
        highlight_function=lambda f: {
            "fillColor": "#ffff00",
            "fillOpacity": 0.3,
            "color": "#333333",
            "weight": 2,
        },
        popup=popup,
        # tooltip removed entirely — click only
    ).add_to(fg)
    fg.add_to(m)
    if colormap:
        colormap.add_to(m)
    print(f"✓ Added '{name}' layer")
    return fg

# ── Load CSV ──────────────────────────────────────────────────────
housing_raw  = None
housing_data = {}
try:
    housing_raw = load_wide_csv(HOUSING_FILE, "Housing/GRAPI")
except FileNotFoundError as e:
    print(f"ERROR: {e}")

# ── Build tract dict and compute burdened_pct BEFORE attaching ────
if housing_raw is not None:
    housing_data = build_tract_dict(housing_raw)

    try:
        for tract_key, row_dict in housing_data.items():
            computed_total = sum(row_dict[r] for r in GRAPI_ROWS if row_dict.get(r) is not None)
            not_computed   = row_dict.get("value_7") or 0.0
            burdened       = row_dict.get("value_6")
            housing_units_total              = int(computed_total + not_computed)
            row_dict['housing_units_total']  = housing_units_total
            row_dict['total']                = float(computed_total) if computed_total > 0 else None
            row_dict['burdened_pct']         = round((burdened / computed_total) * 100, 2) if burdened and computed_total > 0 else None
    except KeyError as e:
        print(f"ERROR computing burdened_pct: {e}")
        print(f"  Available rows: {list(housing_raw.index)}")

    attach_to_features(alachua_features, housing_data, housing_raw.index, 'housing')

    for feature in alachua_features:
        tract_ce = normalize_tract(feature['properties']['TRACTCE'])
        d = housing_data.get(tract_ce, {})
        feature['properties']['grapi_pct']          = d.get('burdened_pct')
        feature['properties']['housing_units_total'] = d.get('housing_units_total')

    grapi_matched = sum(1 for f in alachua_features if f['properties'].get('grapi_pct') is not None)
    print(f"✓ Computed burdened_pct for {grapi_matched}/{len(alachua_features)} tracts")

# ── Colormaps ─────────────────────────────────────────────────────
grapi_colormap = make_colormap(
    alachua_features, 'grapi_pct',
    colors=['#fff7ed','#fed7aa','#fb923c','#ea580c','#c2410c','#7c2d12'],
    caption='% Renters Paying 35%+ of Income on Rent (Rent Burdened)'
)

housing_color_row = "value_1"
housing_colormap  = make_colormap(
    alachua_features, f'housing_{housing_color_row}',
    colors=['#f0f9ff','#bae6fd','#7dd3fc','#38bdf8','#0ea5e9','#0284c7','#0369a1'],
    caption=f'Housing Data — {row_aliases.get(housing_color_row, housing_color_row)}'
)

# ── Color functions ───────────────────────────────────────────────
def get_grapi_color(tract_ce):
    pct = housing_data.get(normalize_tract(tract_ce), {}).get('burdened_pct')
    if pct is None or grapi_colormap is None:
        return '#cccccc'
    return grapi_colormap(pct)

def get_housing_color(tract_ce):
    val = housing_data.get(normalize_tract(tract_ce), {}).get(housing_color_row)
    if val is None or housing_colormap is None:
        return '#cccccc'
    return housing_colormap(val)

# ── Add layer to map ──────────────────────────────────────────────
if housing_data and grapi_colormap:
    grapi_tooltip_fields  = ["TRACTCE", "grapi_pct"] + [f"housing_{r}" for r in housing_raw.index] + ["housing_units_total", "FB", "Total"]
    grapi_tooltip_aliases = ["Tract:", "Rent Burdened (%):"] + [row_aliases.get(r, r) for r in housing_raw.index] + ["Total Renter Units:", "Foreign Born:", "Total Pop:"]
    add_geojson_layer(
        m, geojson_alachua, get_grapi_color,
        grapi_tooltip_fields, grapi_tooltip_aliases,
        "Gross Rent as a % of Household Income", grapi_colormap,
        popup_fields=["TRACTCE", "grapi_pct"] + [f"housing_{r}" for r in housing_raw.index] + ["housing_units_total", "FB", "Total"],
        popup_aliases=["Census Tract:", "Rent Burdened (%):"] + [row_aliases.get(r, r) for r in housing_raw.index] + ["Total Renter Units:", "Foreign Born:", "Total Pop:"],
    )
# =========================
# 13. NO MORTGAGE UNITS LAYER
# =========================

NO_MORTGAGE_FILE = "/content/drive/MyDrive/capstone/no_mortgage.csv"

NO_MORTGAGE_ROWS = ["value_1", "value_2", "value_3", "value_4", "value_5", "value_6", "value_7", "value_8"]

no_mortgage_aliases = {
    "value_1": "< 10% No Mortgage",
    "value_2": "10-14.9% No Mortgage",
    "value_3": "15-19.9% No Mortgage",
    "value_4": "20-24.9% No Mortgage",
    "value_5": "25-29.9% No Mortgage",
    "value_6": "30-34.9% No Mortgage",
    "value_7": "> 35% No Mortgage",
    "value_8": "Not Computed",
}

no_mortgage_raw  = None
no_mortgage_data = {}
no_mortgage_data_normalized = {}

try:
    no_mortgage_raw = load_wide_csv(NO_MORTGAGE_FILE, "% of Households with No Mortgages")
except FileNotFoundError as e:
    print(f"ERROR: {e}")

if no_mortgage_raw is not None:
    no_mortgage_data = build_tract_dict(no_mortgage_raw)

    try:
        for tract_key, row_dict in no_mortgage_data.items():
            full_total   = sum(row_dict[r] for r in NO_MORTGAGE_ROWS if row_dict.get(r) is not None)
            comp_total   = sum(row_dict[r] for r in NO_MORTGAGE_ROWS[:-1] if row_dict.get(r) is not None)
            top_bucket   = row_dict.get("value_7")
            row_dict['nomortgage_units_total'] = int(full_total) if full_total > 0 else 0
            row_dict['total']   = float(comp_total) if comp_total > 0 else None
            row_dict['top_pct'] = round((top_bucket / comp_total) * 100, 2) if top_bucket is not None and comp_total > 0 else (0.0 if top_bucket is not None else None)
    except KeyError as e:
        print(f"ERROR computing no_mortgage top_pct: {e}")
        print(f"  Available rows: {list(no_mortgage_raw.index)}")

    attach_to_features(alachua_features, no_mortgage_data, no_mortgage_raw.index, 'nomortgage')

    no_mortgage_data_normalized = {normalize_tract(k): v for k, v in no_mortgage_data.items()}

    for feature in alachua_features:
        tract_ce = normalize_tract(feature['properties']['TRACTCE'])
        d = no_mortgage_data_normalized.get(tract_ce, {})
        feature['properties']['nomortgage_top_pct']    = d.get('top_pct')
        feature['properties']['nomortgage_units_total'] = d.get('nomortgage_units_total')

    nm_matched = sum(1 for f in alachua_features if f['properties'].get('nomortgage_top_pct') is not None)
    print(f"✓ Computed top_pct for {nm_matched}/{len(alachua_features)} no-mortgage tracts")

    unmatched_nm = [normalize_tract(f['properties']['TRACTCE']) for f in alachua_features if f['properties'].get('nomortgage_top_pct') is None]
    if unmatched_nm:
        print(f"  Unmatched TRACTCE values: {sorted(unmatched_nm)}")

# ── Colormap ──────────────────────────────────────────────────────
no_mortgage_colormap = cm.StepColormap(
    colors=['#f5f0ff','#ddd6fe','#c4b5fd','#a78bfa','#7c3aed','#5b21b6','#3b0764'],
    index=[0, 5, 10, 20, 35, 50, 75, 100],
    vmin=0, vmax=100,
    caption='Owner Units w/ No Mortgage'
)

def get_no_mortgage_color(tract_ce):
    pct = no_mortgage_data_normalized.get(normalize_tract(tract_ce), {}).get('top_pct')
    if pct is None or no_mortgage_colormap is None:
        return '#cccccc'
    return no_mortgage_colormap(pct)

# ── Add layer to map ──────────────────────────────────────────────
if no_mortgage_data and no_mortgage_colormap:
    nm_tooltip_fields  = ["TRACTCE"] + [f"nomortgage_{r}" for r in no_mortgage_raw.index] + ["nomortgage_units_total", "FB", "Total"]
    nm_tooltip_aliases = ["Tract:"] + [no_mortgage_aliases.get(r, r) for r in no_mortgage_raw.index] + ["Total Owned Units with No Mortgage:", "Foreign Born:", "Total Pop:"]
    add_geojson_layer(
        m, geojson_alachua, get_no_mortgage_color,
        nm_tooltip_fields, nm_tooltip_aliases,
        "% Units with No Mortgage", no_mortgage_colormap,
        popup_fields=["TRACTCE", "nomortgage_top_pct"] + [f"nomortgage_{r}" for r in no_mortgage_raw.index] + ["nomortgage_units_total", "FB", "Total"],
        popup_aliases=["Census Tract:", "> 35% Housing Cost Burden (%):"] + [no_mortgage_aliases.get(r, r) for r in no_mortgage_raw.index] + ["Total Owner Units (No Mortgage):", "Foreign Born:", "Total Pop:"],
    )

# =========================
# 14. WITH MORTGAGE UNITS LAYER
# =========================

WITH_MORTGAGE_FILE = "/content/drive/MyDrive/capstone/with_mortgage.csv"

WITH_MORTGAGE_ROWS         = ["value_1", "value_2", "value_3", "value_4", "value_5", "value_6"]
WITH_MORTGAGE_ROWS_COUNTED = ["value_1", "value_2", "value_3", "value_4", "value_5"]

with_mortgage_aliases = {
    "value_1": "< 20% With Mortgage",
    "value_2": "20-24.9% With Mortgage",
    "value_3": "25-29.9% With Mortgage",
    "value_4": "30-34.9% With Mortgage",
    "value_5": "> 35% With Mortgage",
    "value_6": "Not Computed",
    "Housing units with a mortgage (excluding units where SMOCAPI cannot be computed)": "Total Owner Units with Mortgage:",
}

with_mortgage_raw  = None
with_mortgage_data = {}
with_mortgage_data_normalized = {}

try:
    with_mortgage_raw = load_wide_csv(WITH_MORTGAGE_FILE, "With Mortgage Units")
except FileNotFoundError as e:
    print(f"ERROR: {e}")

if with_mortgage_raw is not None:
    with_mortgage_data = build_tract_dict(with_mortgage_raw)

    try:
        for tract_key, row_dict in with_mortgage_data.items():
            full_total  = sum(row_dict[r] for r in WITH_MORTGAGE_ROWS if row_dict.get(r) is not None)
            comp_total  = sum(row_dict[r] for r in WITH_MORTGAGE_ROWS_COUNTED if row_dict.get(r) is not None)
            top_bucket  = row_dict.get("value_5")
            row_dict['with_mortgage_units_total'] = int(full_total) if full_total > 0 else 0
            row_dict['total']   = float(comp_total) if comp_total > 0 else None
            row_dict['top_pct'] = round((top_bucket / comp_total) * 100, 2) if top_bucket is not None and comp_total > 0 else (0.0 if top_bucket is not None else None)
    except KeyError as e:
        print(f"ERROR computing with_mortgage top_pct: {e}")
        print(f"  Available rows: {list(with_mortgage_raw.index)}")

    attach_to_features(alachua_features, with_mortgage_data, with_mortgage_raw.index, 'withmortgage')

    with_mortgage_data_normalized = {normalize_tract(k): v for k, v in with_mortgage_data.items()}

    for feature in alachua_features:
        tract_ce = normalize_tract(feature['properties']['TRACTCE'])
        d = with_mortgage_data_normalized.get(tract_ce, {})
        feature['properties']['withmortgage_top_pct']     = d.get('top_pct')
        feature['properties']['withmortgage_units_total'] = d.get('with_mortgage_units_total')

    wm_matched = sum(1 for f in alachua_features if f['properties'].get('withmortgage_top_pct') is not None)
    print(f"✓ Computed top_pct for {wm_matched}/{len(alachua_features)} with-mortgage tracts")

    unmatched_wm = [normalize_tract(f['properties']['TRACTCE']) for f in alachua_features if f['properties'].get('withmortgage_top_pct') is None]
    if unmatched_wm:
        print(f"  Unmatched TRACTCE values: {sorted(unmatched_wm)}")

# ── Colormap ──────────────────────────────────────────────────────
with_mortgage_colormap = make_colormap(
    alachua_features, 'withmortgage_top_pct',
    colors=['#fff0f9','#fdd0ef','#fba4d8','#f472b6','#db2777','#9d174d','#6b0d36'],
    caption="Owner Units with Mortgage"
)

def get_with_mortgage_color(tract_ce):
    pct = with_mortgage_data_normalized.get(normalize_tract(tract_ce), {}).get('top_pct')
    if pct is None or with_mortgage_colormap is None:
        return '#cccccc'
    return with_mortgage_colormap(pct)

# ── Add layer to map ──────────────────────────────────────────────
if with_mortgage_data and with_mortgage_colormap:
    wm_tooltip_fields  = ["TRACTCE"] + [f"withmortgage_{r}" for r in with_mortgage_raw.index if r in WITH_MORTGAGE_ROWS] + ["withmortgage_units_total", "FB", "Total"]
    wm_tooltip_aliases = ["Tract:"] + [with_mortgage_aliases.get(r, r) for r in with_mortgage_raw.index if r in WITH_MORTGAGE_ROWS] + ["Total Owner Units with Mortgage:", "Foreign Born:", "Total Pop:"]
    add_geojson_layer(
        m, geojson_alachua, get_with_mortgage_color,
        wm_tooltip_fields, wm_tooltip_aliases,
        "% Units with Mortgage", with_mortgage_colormap,
        popup_fields=["TRACTCE", "withmortgage_top_pct"] + [f"withmortgage_{r}" for r in with_mortgage_raw.index if r in WITH_MORTGAGE_ROWS] + ["withmortgage_units_total", "FB", "Total"],
        popup_aliases=["Census Tract:", "> 35% Housing Cost Burden (%):"] + [with_mortgage_aliases.get(r, r) for r in with_mortgage_raw.index if r in WITH_MORTGAGE_ROWS] + ["Total Owner Units (Mortgage):", "Foreign Born:", "Total Pop:"],
    )

# =========================
# 15. TITLE, COMPASS, SCALE
# =========================

title_html = '''
<div style="position:fixed; top:15px; left:50%; transform:translateX(-50%);
            z-index:9999; background:rgba(255,255,255,0.9); border:2px solid #2c3e50;
            border-radius:8px; padding:10px 20px; box-shadow:0 3px 8px rgba(0,0,0,0.4);
            font-family:Cambria,sans-serif; font-size:20px; font-weight:bold;
            color:#2c3e50; text-align:center;">
    Gainesville Comprehensive Map
</div>'''
m.get_root().html.add_child(folium.Element(title_html))

compass_html = '''
<div style="position:fixed; top:100px; left:15px; width:70px; height:70px; z-index:9999;
            background:radial-gradient(circle,rgba(255,255,255,0.95) 0%,rgba(240,240,240,0.9) 100%);
            border:2.5px solid #2c3e50; border-radius:50%; box-shadow:0 3px 8px rgba(0,0,0,0.4);
            display:flex; align-items:center; justify-content:center; font-family:Arial,sans-serif;">
  <div style="position:relative; width:100%; height:100%;">
    <div style="position:absolute; top:50%; left:50%; width:0; height:0;
                border-left:6px solid transparent; border-right:6px solid transparent;
                border-bottom:24px solid #c0392b;
                transform:translate(-50%,-100%) translateY(6px);"></div>
    <div style="position:absolute; top:50%; left:50%; width:0; height:0;
                border-left:6px solid transparent; border-right:6px solid transparent;
                border-top:24px solid #7f8c8d;
                transform:translate(-50%,0%) translateY(-6px);"></div>
    <div style="position:absolute; top:50%; left:50%; width:0; height:0;
                border-top:5px solid transparent; border-bottom:5px solid transparent;
                border-left:20px solid #95a5a6;
                transform:translate(0%,-50%) translateX(-6px);"></div>
    <div style="position:absolute; top:50%; left:50%; width:0; height:0;
                border-top:5px solid transparent; border-bottom:5px solid transparent;
                border-right:20px solid #95a5a6;
                transform:translate(-100%,-50%) translateX(6px);"></div>
    <div style="position:absolute; top:50%; left:50%; width:8px; height:8px;
                background:#34495e; border-radius:50%; transform:translate(-50%,-50%);"></div>
    <div style="position:absolute; top:3px; left:50%; transform:translateX(-50%);
                color:#c0392b; font-weight:bold; font-size:13px;">N</div>
    <div style="position:absolute; bottom:3px; left:50%; transform:translateX(-50%);
                color:#7f8c8d; font-weight:bold; font-size:11px;">S</div>
    <div style="position:absolute; right:3px; top:50%; transform:translateY(-50%);
                color:#95a5a6; font-weight:bold; font-size:11px;">E</div>
    <div style="position:absolute; left:3px; top:50%; transform:translateY(-50%);
                color:#95a5a6; font-weight:bold; font-size:11px;">W</div>
  </div>
</div>'''
m.get_root().html.add_child(folium.Element(compass_html))

from folium.plugins import MeasureControl
m.add_child(MeasureControl(
    position='bottomleft',
    primary_length_unit='miles', secondary_length_unit='meters',
    primary_area_unit='sqmiles', secondary_area_unit='sqmeters'
))

# =========================
# 16. TRACT INFO CLICK LAYER
# =========================

fg_tooltip = folium.FeatureGroup(name="Tract Info (Click)", show=True, overlay=True, control=False)

folium.GeoJson(
    geojson_alachua,
    style_function=lambda f: {
        "fillColor": "transparent",
        "fillOpacity": 0,
        "color": "transparent",
        "weight": 0,
    },
    highlight_function=lambda f: {
        "fillColor": "#ffff00",
        "fillOpacity": 0.2,
        "color": "#333333",
        "weight": 2,
    },
    popup=folium.GeoJsonPopup(
        fields=[
            "TRACTCE",
            "Total",
            "FB",
            "FB_pct",
            "veh_pct",
            "broadband_pct",
            "computer_pct",
        ],
        aliases=[
            "Census Tract:",
            "Total Population:",
            "Foreign Born (count):",
            "Foreign Born (%):",
            "Households with Vehicle (%):",
            "Households with Broadband (%):",
            "Households with Computer (%):",
        ],
        localize=True,
        sticky=False,
        style=(
            "background-color: white;"
            "border: 1px solid #333;"
            "border-radius: 4px;"
            "padding: 8px;"
            "font-family: Arial;"
            "font-size: 13px;"
        ),
        max_width=300,
    ),
).add_to(fg_tooltip)

## =========================
# 17. LAYER CONTROL AND SAVE
# =========================

tooltip_on_top_js = """
<script>
document.addEventListener("DOMContentLoaded", function() {
    var tooltipPane = document.querySelector('.leaflet-tooltip-pane');
    if (tooltipPane) {
        tooltipPane.style.zIndex = 9999;
    }
});
</script>
"""

boundary_on_top_js = """
<script>
document.addEventListener("DOMContentLoaded", function() {
    setTimeout(function() {
        var map = null;
        for (var key in window) {
            try {
                if (window[key] && window[key]._leaflet_id && window[key].eachLayer) {
                    map = window[key];
                    break;
                }
            } catch(e) {}
        }
        if (!map) return;

        var topColors = ['#ff0000', '#ff6600', '#e67e22', '#27ae60', '#2980b9', '#8e44ad'];

        function bringBoundariesToFront() {
            map.eachLayer(function(layer) {
                if (layer.options && topColors.indexOf(layer.options.color) !== -1) {
                    layer.bringToFront();
                }
                if (layer.eachLayer) {
                    layer.eachLayer(function(sublayer) {
                        if (sublayer.options && topColors.indexOf(sublayer.options.color) !== -1) {
                            sublayer.bringToFront();
                        }
                        if (sublayer.eachLayer) {
                            sublayer.eachLayer(function(subsublayer) {
                                if (subsublayer.options && topColors.indexOf(subsublayer.options.color) !== -1) {
                                    subsublayer.bringToFront();
                                }
                            });
                        }
                    });
                }
            });
        }

        bringBoundariesToFront();

        map.on('layeradd layerremove overlayadd overlayremove', function() {
            setTimeout(bringBoundariesToFront, 100);
        });

    }, 1000);
});
</script>
"""

colormap_fix_js = """
<script>
document.addEventListener("DOMContentLoaded", function() {
    setTimeout(function() {

        // Hide all branca colormaps
        var colormaps = document.querySelectorAll('.leaflet-control.legend');
        colormaps.forEach(function(el) {
            el.style.setProperty('display', 'none', 'important');
        });

        // Gradient legend definitions
        var legends = [
            { label: 'Foreign Born Population',             colors: ['#f0f9ff','#bae6fd','#7dd3fc','#38bdf8','#0ea5e9','#0284c7','#0369a1','#075985','#0c4a6e'], min: '19',  max: '1,907', hatch: false },
            { label: '% Households with At Least One Vehicle',   colors: ['#f0f4f0','#daeada','#bcd8bc','#97c297','#72aa72','#4e914e','#347834','#266026','#1a6b1a'], min: '50%', max: '100%',  hatch: false },
            { label: '% Households with Broadband Access', colors: ['#ffffb7','#fff192','#ffea61','#ffdd3c','#ffd400','#c29200'], min: '30%', max: '100%', hatch: true },
            { label: '% Renters Rent Burdened',           colors: ['#fff7ed','#fed7aa','#fb923c','#ea580c','#c2410c','#7c2d12'],                               min: '0%',  max: '100%',  hatch: false },
            { label: 'Owner Units w/ No Mortgage', colors: ['#f5f0ff','#ddd6fe','#c4b5fd','#a78bfa','#7c3aed','#5b21b6','#3b0764'],                   min: '0%',  max: '100%',  hatch: false },
            { label: 'Owner Units w/ Mortgage',    colors: ['#fff0f9','#fdd0ef','#fba4d8','#f472b6','#db2777','#9d174d','#6b0d36'],                   min: '0%',  max: '100%',  hatch: false },
        ];

        // Single container stacked vertically on the left
        var container = document.createElement('div');
        container.style.cssText = [
            'position:fixed',
            'left:10px',
            'top:50%',
            'transform:translateY(-50%)',
            'z-index:9999',
            'display:flex',
            'flex-direction:column',
            'gap:6px',
        ].join(';');

        // Build gradient legend items
        legends.forEach(function(leg) {
            var wrapper = document.createElement('div');
            wrapper.style.cssText = 'display:flex;flex-direction:column;gap:2px;';

            var div = document.createElement('div');
            div.style.cssText = [
                'background:rgba(255,255,255,0.9)',
                'padding:4px 8px',
                'border-radius:4px',
                'box-shadow:0 1px 4px rgba(0,0,0,0.3)',
                'font-family:Arial',
                'font-size:11px',
                'white-space:nowrap',
            ].join(';');

            var gradient = 'linear-gradient(to right, ' + leg.colors.join(',') + ')';
            div.innerHTML =
                '<div style="margin-bottom:2px;font-weight:bold;">' + leg.label + '</div>' +
                '<div style="display:flex;align-items:center;gap:4px;">' +
                    '<span>' + leg.min + '</span>' +
                    '<div style="width:160px;height:12px;background:' + gradient + ';border-radius:2px;"></div>' +
                    '<span>' + leg.max + '</span>' +
                '</div>';

            wrapper.appendChild(div);

            if (leg.hatch) {
                var hatchDiv = document.createElement('div');
                hatchDiv.style.cssText = [
                    'background:rgba(255,255,255,0.9)',
                    'padding:4px 8px',
                    'border-radius:4px',
                    'box-shadow:0 1px 4px rgba(0,0,0,0.3)',
                    'font-family:Arial',
                    'font-size:11px',
                    'white-space:nowrap',
                    'margin-left:16px',
                    'border-left:3px solid #ef4444',
                ].join(';');

                hatchDiv.innerHTML =
                    '<div style="margin-bottom:4px;font-weight:bold;color:#555;">↳ Hatching = % Households with Computer</div>' +
                    '<div style="display:flex;flex-direction:column;gap:4px;">' +
                        '<div style="display:flex;align-items:center;gap:6px;">' +
                            '<div style="width:30px;height:12px;background:repeating-linear-gradient(45deg,#000,#000 1px,transparent 1px,transparent 4px);border:1px solid #ccc;"></div>' +
                            '<span>Dense = Higher computer access</span>' +
                        '</div>' +
                        '<div style="display:flex;align-items:center;gap:6px;">' +
                            '<div style="width:30px;height:12px;background:repeating-linear-gradient(45deg,#000,#000 1px,transparent 1px,transparent 12px);border:1px solid #ccc;"></div>' +
                            '<span>Sparse = Lower computer access</span>' +
                        '</div>' +
                    '</div>';

                wrapper.appendChild(hatchDiv);
            }

            container.appendChild(wrapper);
        });

        document.body.appendChild(container);

    }, 1500);
});
</script>
"""

fg_tooltip.add_to(m)

m.get_root().html.add_child(folium.Element(tooltip_on_top_js))
m.get_root().html.add_child(folium.Element(boundary_on_top_js))
m.get_root().html.add_child(folium.Element(colormap_fix_js))

# ── City limits (not toggleable, always visible) ──
if gainesville_geom:
    folium.GeoJson(
        {"type": "Feature", "geometry": gainesville_geom,
         "properties": {"name": "Gainesville",
                        "sq_mi": f"{gainesville_acres/640:.1f}" if gainesville_acres else "N/A"}},
        style_function=lambda f: {
            "fillColor": "none", "color": "#ff0000", "weight": 3, "fillOpacity": 0,
        },
        tooltip=folium.GeoJsonTooltip(
            fields=["name", "sq_mi"], aliases=["City:", "Area (sq mi):"], sticky=True
        ),
        overlay=True,
        control=False,
        show=True,
    ).add_to(m)
    print("✓ Added Gainesville city boundary (always on top)")

# ── Commission districts as toggleable layer ──
try:
    import copy
    commission_geometries = read_shp(COMMISSION_SHP)
    commission_records, _ = read_dbf(COMMISSION_DBF)

    commission_fg = folium.FeatureGroup(name="City Commission Districts", show=True)

    for i, record in enumerate(commission_records):
        if not commission_geometries[i]:
            continue

        geom = copy.deepcopy(commission_geometries[i])

        if geom.get('needs_transform', False):
            if geom['type'] == 'Polygon':
                geom['coordinates'] = [
                    [list(stateplane_to_latlon(x, y)) for x, y in ring]
                    for ring in geom['coordinates']
                ]
            elif geom['type'] == 'MultiPolygon':
                geom['coordinates'] = [
                    [
                        [list(stateplane_to_latlon(x, y)) for x, y in ring]
                        for ring in polygon
                    ]
                    for polygon in geom['coordinates']
                ]
            del geom['needs_transform']

        district_id   = record.get('DISTRICT_I', i + 1)
        district_name = record.get('DISTRICT_N', f'District {district_id}')
        rep_name      = record.get('REP_NAME', 'N/A')

        folium.GeoJson(
            {
                "type": "Feature",
                "geometry": geom,
                "properties": {
                    "district_id":   str(district_id),
                    "district_name": district_name,
                    "rep_name":      rep_name,
                },
            },
            style_function=lambda f: {
                "fillColor":   "none",
                "fillOpacity": 0,
                "color":       "#ff0000",
                "weight":      3,
            },
            tooltip=folium.GeoJsonTooltip(
                fields=["district_name", "district_id", "rep_name"],
                aliases=["District:", "District #:", "Representative:"],
                sticky=True,
            ),
        ).add_to(commission_fg)

    commission_fg.add_to(m)
    print(f"✓ Added {len(commission_records)} City Commission District boundaries (toggleable)")

except Exception as e:
    import traceback
    print(f"WARNING: Could not load City Commission Districts — {e}")
    traceback.print_exc()

# ── Add LayerControl LAST so all feature groups are registered ──
folium.LayerControl(collapsed=False).add_to(m)

output_path = "/content/drive/MyDrive/capstone/gbm_complete.html"
m.save(output_path)

Loading data...
✓ Loaded 974 bus stops
✓ Loaded 52 bus route segments (26 unique routes)
  Routes: ['1', '10', '11', '118', '12', '13', '15', '17', '20', '21', '23', '26', '3', '33', '37', '38', '43', '5', '52', '55', '6', '7', '75', '76', '8', '9']
✓ Loaded 58 census tracts with FB data
✓ Loaded Gainesville city limits (41511.8 acres = 64.9 sq mi)
✓ Loaded 12 library locations
✓ Loaded vehicle data for 58 census tracts
  Tract 2.01: 78.86% with at least one vehicle
  Tract 2.02: 94.5% with at least one vehicle
  Tract 3.01: 77.22% with at least one vehicle
  Sample vehicle_data keys: ['2.01', '2.02', '3.01', '3.02', '4']

Reading Alachua County shapefile...
✓ Found 5160 census tracts in Florida shapefile
✓ Filtered to 58 Alachua County census tracts
✓ Matched 58 tracts with FB data
✓ Matched 58 tracts with vehicle data
✓ Added Vehicle layer
✓ Loaded internet/computer data for 58 tracts
✓ Matched 58/58 tracts with internet data
  Unmatched TRACTCE values: []
  Internet CSV keys (all): 